In [1]:
# =============================================================================
# SECTION 1 : CONFIGURATION
# =============================================================================

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# =============================================================================
# Reproducibility
# =============================================================================

RANDOM_STATE = 42

# =============================================================================
# AutoML Search Configuration
# =============================================================================

# Time budget for ONE Experiment Space
EXPERIMENT_SPACE_BUDGET = 60 * 60      # seconds (1 hour)

# Maximum time for evaluating one candidate pipeline
PER_RUN_TIME_LIMIT = 300

# =============================================================================
# Pipeline Selection
# =============================================================================

# Pipelines within ±1% of the baseline accuracy are retained
ACCURACY_THRESHOLD = 0.01

# =============================================================================
# SHAP Configuration
# =============================================================================

BACKGROUND_SAMPLES = 100
TEST_SAMPLES = 50

# =============================================================================
# Parallel Execution
# =============================================================================

NUM_PARALLEL_DATASETS = 6

# =============================================================================
# Output Directory
# =============================================================================

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print("=" * 80)
print("SECTION 1 : CONFIGURATION")
print("=" * 80)

print(f"Random State               : {RANDOM_STATE}")
print(f"Experiment Space Budget    : {EXPERIMENT_SPACE_BUDGET//3600} hour")
print(f"Per Run Time Limit         : {PER_RUN_TIME_LIMIT} sec")
print(f"Accuracy Threshold         : ±{ACCURACY_THRESHOLD*100:.1f}%")
print(f"Background Samples         : {BACKGROUND_SAMPLES}")
print(f"Test Samples               : {TEST_SAMPLES}")
print(f"Parallel Dataset Workers   : {NUM_PARALLEL_DATASETS}")
print(f"Results Directory          : {RESULTS_DIR}")

SECTION 1 : CONFIGURATION
Random State               : 42
Experiment Space Budget    : 1 hour
Per Run Time Limit         : 300 sec
Accuracy Threshold         : ±1.0%
Background Samples         : 100
Test Samples               : 50
Parallel Dataset Workers   : 6
Results Directory          : results


In [2]:
# =============================================================================
# SECTION 2 : IMPORTS
# =============================================================================

# ---------- Standard Library ----------
import os
import json
import pickle
import itertools
import inspect
from pathlib import Path

# ---------- Data ----------
import numpy as np
import pandas as pd

# ---------- Visualization ----------
import matplotlib.pyplot as plt

# ---------- OpenML ----------
import openml

# ---------- SHAP ----------
import shap


from shap_rf_and_autosklearn import ClassificationAlgorithms
from Importance import FeatureImportance

print("=" * 80)
print("SECTION 2 : IMPORTS")
print("=" * 80)
print("✓ All imports loaded successfully.")

SECTION 2 : IMPORTS
✓ All imports loaded successfully.


In [3]:
# =============================================================================
# SECTION 3 : SEARCH SPACE DISCOVERY
# =============================================================================

from autosklearn.pipeline.components.classification import (
    _classifiers as CLASSIFIERS
)

classifier_names = sorted(CLASSIFIERS.keys())

print("=" * 80)
print("AVAILABLE CLASSIFIERS")
print("=" * 80)

for i, name in enumerate(classifier_names, start=1):
    print(f"{i:2d}. {name}")

print("\nTotal Classifiers :", len(classifier_names))


# =============================================================================
#  FEATURE PREPROCESSOR DISCOVERY
# =============================================================================

from autosklearn.pipeline.components.feature_preprocessing import (
    _preprocessors as PREPROCESSORS
)

preprocessor_names = sorted(PREPROCESSORS.keys())

print("=" * 80)
print("AVAILABLE FEATURE PREPROCESSORS")
print("=" * 80)

for i, name in enumerate(preprocessor_names, start=1):
    print(f"{i:2d}. {name}")

print("\nTotal Feature Preprocessors :", len(preprocessor_names))

AVAILABLE CLASSIFIERS
 1. adaboost
 2. bernoulli_nb
 3. decision_tree
 4. extra_trees
 5. gaussian_nb
 6. gradient_boosting
 7. k_nearest_neighbors
 8. lda
 9. liblinear_svc
10. libsvm_svc
11. mlp
12. multinomial_nb
13. passive_aggressive
14. qda
15. random_forest
16. sgd

Total Classifiers : 16
AVAILABLE FEATURE PREPROCESSORS
 1. densifier
 2. extra_trees_preproc_for_classification
 3. extra_trees_preproc_for_regression
 4. fast_ica
 5. feature_agglomeration
 6. kernel_pca
 7. kitchen_sinks
 8. liblinear_svc_preprocessor
 9. no_preprocessing
10. nystroem_sampler
11. pca
12. polynomial
13. random_trees_embedding
14. select_percentile_classification
15. select_percentile_regression
16. select_rates_classification
17. select_rates_regression
18. truncatedSVD

Total Feature Preprocessors : 18


In [4]:
# =============================================================================
# SECTION 4 : EXPERIMENT SPACE GENERATION
# =============================================================================

experiment_spaces = pd.DataFrame(
    list(itertools.product(classifier_names, preprocessor_names)),
    columns=["classifier", "feature_preprocessor"]
)

experiment_spaces.insert(
    0,
    "experiment_id",
    range(1, len(experiment_spaces) + 1)
)

print("=" * 80)
print("EXPERIMENT SPACE SUMMARY")
print("=" * 80)

print(f"Total Classifiers            : {len(classifier_names)}")
print(f"Total Feature Preprocessors  : {len(preprocessor_names)}")
print(f"Total Experiment Spaces      : {len(experiment_spaces)}")

print("\nFirst 10 Experiment Spaces\n")
display(experiment_spaces.head(10))

EXPERIMENT SPACE SUMMARY
Total Classifiers            : 16
Total Feature Preprocessors  : 18
Total Experiment Spaces      : 288

First 10 Experiment Spaces



,experiment_id,classifier,feature_preprocessor
0,1,adaboost,densifier
1,2,adaboost,extra_trees_preproc_for_classification
2,3,adaboost,extra_trees_preproc_for_regression
3,4,adaboost,fast_ica
4,5,adaboost,feature_agglomeration
5,6,adaboost,kernel_pca
6,7,adaboost,kitchen_sinks
7,8,adaboost,liblinear_svc_preprocessor
8,9,adaboost,no_preprocessing
9,10,adaboost,nystroem_sampler


In [17]:
# =============================================================================
# SECTION 5 : DATASET SELECTION
# =============================================================================

DATASET_IDS = [
    31,     # credit-g
    37,     # diabetes
    44,     # spambase
    1461,   # bank-marketing
    4534,   # phishing-websites
    151     # electricity
]

dataset_table = pd.DataFrame({
    "dataset_id": DATASET_IDS
})

dataset_table["status"] = "Not Loaded"
dataset_table["n_samples"] = np.nan
dataset_table["n_features"] = np.nan
dataset_table["n_classes"] = np.nan

print("=" * 80)
print("DATASET SUMMARY")
print("=" * 80)

display(dataset_table)

DATASET SUMMARY


,dataset_id,status,n_samples,n_features,n_classes
0,31,Not Loaded,NaN,NaN,NaN
1,37,Not Loaded,NaN,NaN,NaN
2,44,Not Loaded,NaN,NaN,NaN
3,1461,Not Loaded,NaN,NaN,NaN
4,4534,Not Loaded,NaN,NaN,NaN
5,151,Not Loaded,NaN,NaN,NaN


In [18]:
# =============================================================================
# SECTION 5 : LOAD OPENML DATASETS
# =============================================================================

datasets = {}

print("=" * 100)
print("LOADING OPENML DATASETS")
print("=" * 100)

for idx, did in enumerate(DATASET_IDS):

    dataset = openml.datasets.get_dataset(did)

    X, y, categorical_indicator, feature_names = dataset.get_data(
        dataset_format="dataframe",
        target=dataset.default_target_attribute
    )

    datasets[did] = {
        "dataset": dataset,
        "X": X,
        "y": y,
        "categorical_indicator": categorical_indicator,
        "feature_names": feature_names
    }

    dataset_table.loc[idx, "status"] = "Loaded"
    dataset_table.loc[idx, "n_samples"] = X.shape[0]
    dataset_table.loc[idx, "n_features"] = X.shape[1]
    dataset_table.loc[idx, "n_classes"] = y.nunique()

print("All datasets loaded successfully.\n")

display(dataset_table)

LOADING OPENML DATASETS
All datasets loaded successfully.



,dataset_id,status,n_samples,n_features,n_classes
0,31,Loaded,1000.0,20.0,2.0
1,37,Loaded,768.0,8.0,2.0
2,44,Loaded,4601.0,57.0,2.0
3,1461,Loaded,45211.0,16.0,2.0
4,4534,Loaded,11055.0,30.0,2.0
5,151,Loaded,45312.0,8.0,2.0


In [19]:
# =============================================================================
# SECTION 6 : CONVERT OPENML DATASETS into Suitable FORMAT
# =============================================================================

DATASET_ROOT = Path("datasets")
DATASET_ROOT.mkdir(exist_ok=True)

print("=" * 100)
print("CONVERTING OPENML DATASETS")
print("=" * 100)

for did in DATASET_IDS:

    dataset_folder = DATASET_ROOT / str(did)
    dataset_folder.mkdir(parents=True, exist_ok=True)

    X = datasets[did]["X"]
    y = datasets[did]["y"]
    categorical_indicator = datasets[did]["categorical_indicator"]

    # -----------------------------
    # Save Features
    # -----------------------------
    X.to_csv(
        dataset_folder / "features.csv",
        index=False
    )

    # -----------------------------
    # Save Targets
    # -----------------------------
    pd.DataFrame(y).to_csv(
        dataset_folder / "targets.csv",
        index=False
    )

    # -----------------------------
    # Save Categorical Indicator
    # -----------------------------
    categorical_dict = {
        column: indicator
        for column, indicator in zip(X.columns, categorical_indicator)
    }

    with open(dataset_folder / "categorical_indicator.json", "w") as f:
        json.dump(categorical_dict, f, indent=4)

    print(f"✓ Dataset {did} saved to {dataset_folder}")

print("\n✅ All datasets converted successfully.")

CONVERTING OPENML DATASETS
✓ Dataset 31 saved to datasets/31
✓ Dataset 37 saved to datasets/37
✓ Dataset 44 saved to datasets/44
✓ Dataset 1461 saved to datasets/1461
✓ Dataset 4534 saved to datasets/4534
✓ Dataset 151 saved to datasets/151

✅ All datasets converted successfully.


In [8]:
# =============================================================================
# SECTION 7 : BASELINE RANDOM FOREST GENERATION
# =============================================================================

import json
import pickle

BASELINE_RESULTS_DIR = RESULTS_DIR / "baselines"
BASELINE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

baseline_results = {}

print("=" * 100)
print("GENERATING BASELINE RANDOM FOREST MODELS")
print("=" * 100)

for did in DATASET_IDS:

    print(f"\nDataset {did}")

    dataset_path = str(DATASET_ROOT / str(did))

    # -------------------------------------------------------------------------
    # Create folder for this dataset
    # -------------------------------------------------------------------------

    dataset_result_dir = BASELINE_RESULTS_DIR / str(did)
    dataset_result_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------------------------------------------------------
    # Prepare data
    # -------------------------------------------------------------------------

    X_train, X_test, y_train, y_test, feature_names_rf, nan_indices = (
        ClassificationAlgorithms.prepare_data_for_RF_importance(
            datapath=dataset_path,
            randomState=RANDOM_STATE
        )
    )

    # -------------------------------------------------------------------------
    # Train baseline Random Forest
    # -------------------------------------------------------------------------

    baseline_model, baseline_accuracy, dataset_id = (
        ClassificationAlgorithms.train_default_RandomForest_Classifier(
            datasetId=did,
            randomState=RANDOM_STATE,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test
        )
    )

    # -------------------------------------------------------------------------
    # Store in memory
    # -------------------------------------------------------------------------

    baseline_results[did] = {
        "model": baseline_model,
        "accuracy": baseline_accuracy,
        "feature_names": feature_names_rf,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "nan_indices": nan_indices
    }

    # -------------------------------------------------------------------------
    # Save Model
    # -------------------------------------------------------------------------

    with open(dataset_result_dir / "baseline_model.pkl", "wb") as f:
        pickle.dump(baseline_model, f)

    # -------------------------------------------------------------------------
    # Save Metrics
    # -------------------------------------------------------------------------

    metrics = {
        "dataset_id": did,
        "baseline_accuracy": float(baseline_accuracy),
        "n_train_samples": int(X_train.shape[0]),
        "n_test_samples": int(X_test.shape[0]),
        "n_features": int(X_train.shape[1]),
    }

    with open(dataset_result_dir / "baseline_metrics.json", "w") as f:
        json.dump(metrics, f, indent=4)

    print(f"✓ Baseline Accuracy : {baseline_accuracy:.4f}")
    print(f"✓ Saved to : {dataset_result_dir}")

print("\n" + "=" * 100)
print("BASELINE GENERATION COMPLETED")
print("=" * 100)

GENERATING BASELINE RANDOM FOREST MODELS

Dataset 31
✓ Baseline Accuracy : 0.7950
✓ Saved to : results/baselines/31

Dataset 37
✓ Baseline Accuracy : 0.7208
✓ Saved to : results/baselines/37

Dataset 44
✓ Baseline Accuracy : 0.9555
✓ Saved to : results/baselines/44

Dataset 1461
✓ Baseline Accuracy : 0.9005
✓ Saved to : results/baselines/1461

Dataset 4534
✓ Baseline Accuracy : 0.9670
✓ Saved to : results/baselines/4534

Dataset 151
✓ Baseline Accuracy : 0.9062
✓ Saved to : results/baselines/151

BASELINE GENERATION COMPLETED


In [9]:
# =============================================================================
# SECTION 8 : BASELINE SHAP MODEL GENERATION
# =============================================================================

BASELINE_SHAP_DIR = RESULTS_DIR / "baseline_shap"
BASELINE_SHAP_DIR.mkdir(parents=True, exist_ok=True)

baseline_shap_models = {}

print("=" * 100)
print("GENERATING BASELINE SHAP MODELS")
print("=" * 100)

for did in DATASET_IDS:

    print(f"\nDataset {did}")

    dataset_path = str(DATASET_ROOT / str(did))

    features = pd.read_csv(f"{dataset_path}/features.csv")
    targets = pd.read_csv(f"{dataset_path}/targets.csv")

    with open(f"{dataset_path}/categorical_indicator.json", "r") as f:
        cat_indicators = json.load(f)

    (
        X_train,
        X_test,
        y_train,
        y_test,
        X_cat,
        original_feature_names,
        original_cat_feature_names,
        shap_all_feature_names,
        shap_cat_feature_names,
        classes,
        nan_indices
    ) = ClassificationAlgorithms.prepare_data_for_SHAP_importance(
        X=features,
        y=targets,
        datapath=dataset_path,
        cat_indicators=cat_indicators,
        randomState=RANDOM_STATE,
        datasetId=did
    )

    rf_model, accuracy, _ = (
        ClassificationAlgorithms.train_default_RandomForest_Classifier(
            datasetId=did,
            randomState=RANDOM_STATE,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test
        )
    )

    baseline_shap_models[did] = {
        "model": rf_model,
        "accuracy": accuracy,
        "X_train": X_train,
        "X_test": X_test,
        "X_cat": X_cat,
        "original_feature_names": original_feature_names,
        "original_cat_feature_names": original_cat_feature_names,
        "shap_all_feature_names": shap_all_feature_names,
        "shap_cat_feature_names": shap_cat_feature_names,
    }

    print(f"✓ SHAP RF Accuracy : {accuracy:.4f}")

print("\n✅ Baseline SHAP models generated.")

GENERATING BASELINE SHAP MODELS

Dataset 31
(800, 61) (800,) (200, 61) (200,)
✓ SHAP RF Accuracy : 0.7800

Dataset 37
(614, 8) (614,) (154, 8) (154,)
✓ SHAP RF Accuracy : 0.7208

Dataset 44
(3680, 57) (3680,) (921, 57) (921,)
✓ SHAP RF Accuracy : 0.9555

Dataset 1461
(36168, 51) (36168,) (9043, 51) (9043,)
✓ SHAP RF Accuracy : 0.9034

Dataset 4534
(8844, 68) (8844,) (2211, 68) (2211,)
✓ SHAP RF Accuracy : 0.9665

Dataset 151
(36249, 14) (36249,) (9063, 14) (9063,)
✓ SHAP RF Accuracy : 0.9050

✅ Baseline SHAP models generated.


In [23]:
# =============================================================================
# SECTION 9 : COMPUTE BASELINE SHAP
# =============================================================================

BASELINE_SHAP_RESULTS_DIR = RESULTS_DIR / "baseline_shap_results"
BASELINE_SHAP_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

baseline_shap_results = {}

print("=" * 100)
print("COMPUTING BASELINE SHAP")
print("=" * 100)

for did in DATASET_IDS:

    print(f"\nDataset {did}")

    model_data = baseline_shap_models[did]

    (
        abs_mean_shap,
        one_class_shap,
        subcategory_shap,
        one_class_subcategory_shap
    ) = FeatureImportance.get_abs_SHAP_feature_importance(
        classifier=model_data["model"],
        X_train=model_data["X_train"],
        X_test=model_data["X_test"],
        X_cat=model_data["X_cat"],
        original_featureNames=model_data["original_feature_names"],
        original_catFeatureNames=model_data["original_cat_feature_names"],
        shap_allfeatureNames=model_data["shap_all_feature_names"],
        shap_catFeatureNames=model_data["shap_cat_feature_names"]
    )

    baseline_shap_results[did] = {
        "abs_mean_shap": abs_mean_shap,
        "one_class_shap": one_class_shap,
        "subcategory_shap": subcategory_shap,
        "one_class_subcategory_shap": one_class_subcategory_shap,
    }

    dataset_dir = BASELINE_SHAP_RESULTS_DIR / str(did)
    dataset_dir.mkdir(parents=True, exist_ok=True)

    with open(dataset_dir / "baseline_shap.json", "w") as f:
        json.dump(
            {
                "abs_mean_shap": abs_mean_shap,
                "one_class_shap": one_class_shap,
                "subcategory_shap": subcategory_shap,
                "one_class_subcategory_shap": one_class_subcategory_shap,
            },
            f,
            indent=4,
        )

    print(f"✓ Baseline SHAP saved for Dataset {did}")

print("\n" + "=" * 100)
print("BASELINE SHAP COMPLETED")
print("=" * 100)

COMPUTING BASELINE SHAP

Dataset 31


  0%|          | 0/200 [00:00<?, ?it/s]

✓ Baseline SHAP saved for Dataset 31

Dataset 37


  0%|          | 0/154 [00:00<?, ?it/s]

✓ Baseline SHAP saved for Dataset 37

Dataset 44


...

✓ Baseline SHAP saved for Dataset 1461

Dataset 4534


  0%|          | 0/2211 [00:00<?, ?it/s]

✓ Baseline SHAP saved for Dataset 4534

Dataset 151


  0%|          | 0/9063 [00:00<?, ?it/s]

✓ Baseline SHAP saved for Dataset 151

BASELINE SHAP COMPLETED


In [10]:
# =============================================================================
# SECTION 10 : EXPERIMENT CONFIGURATION
# =============================================================================

experiment_jobs = []

for dataset_id in DATASET_IDS:

    for _, row in experiment_spaces.iterrows():

        experiment_jobs.append({

            "dataset_id": dataset_id,

            "experiment_id": row["experiment_id"],

            "classifier": row["classifier"],

            "feature_preprocessor": row["feature_preprocessor"],

            "time_budget": EXPERIMENT_SPACE_BUDGET,

            "per_run_time_limit": PER_RUN_TIME_LIMIT,

            "status": "Pending"

        })

experiment_jobs = pd.DataFrame(experiment_jobs)

print("=" * 100)
print("EXPERIMENT JOB SUMMARY")
print("=" * 100)

print(f"Datasets              : {len(DATASET_IDS)}")
print(f"Experiment Spaces     : {len(experiment_spaces)}")
print(f"Total Jobs            : {len(experiment_jobs)}")

display(experiment_jobs.head())

EXPERIMENT JOB SUMMARY
Datasets              : 6
Experiment Spaces     : 288
Total Jobs            : 1728


,dataset_id,experiment_id,classifier,feature_preprocessor,time_budget,per_run_time_limit,status
0,31,1,adaboost,densifier,3600,300,Pending
1,31,2,adaboost,extra_trees_preproc_for_classification,3600,300,Pending
2,31,3,adaboost,extra_trees_preproc_for_regression,3600,300,Pending
3,31,4,adaboost,fast_ica,3600,300,Pending
4,31,5,adaboost,feature_agglomeration,3600,300,Pending


In [11]:
# =============================================================================
# SECTION 11 : CREATE EXPERIMENT DIRECTORIES
# =============================================================================

from pathlib import Path
import json

EXPERIMENT_ROOT = RESULTS_DIR / "experiments"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("CREATING EXPERIMENT DIRECTORIES")
print("=" * 100)

for _, row in experiment_jobs.iterrows():

    dataset_id = int(row["dataset_id"])
    experiment_id = int(row["experiment_id"])

    experiment_dir = (
        EXPERIMENT_ROOT
        / f"dataset_{dataset_id}"
        / f"experiment_{experiment_id:03d}"
    )

    experiment_dir.mkdir(parents=True, exist_ok=True)

    metadata = {
        "dataset_id": dataset_id,
        "experiment_id": experiment_id,
        "classifier": row["classifier"],
        "feature_preprocessor": row["feature_preprocessor"],
        "time_budget": row["time_budget"],
        "per_run_time_limit": row["per_run_time_limit"],
        "random_state": RANDOM_STATE,
        "status": "Pending"
    }

    with open(experiment_dir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=4)

print(f"Created {len(experiment_jobs)} experiment directories.")

CREATING EXPERIMENT DIRECTORIES
Created 1728 experiment directories.


In [12]:
# =============================================================================
# RECREATE SEARCH SPACE
# =============================================================================

classifiers = [
    'adaboost',
    'bernoulli_nb',
    'decision_tree',
    'extra_trees',
    'gaussian_nb',
    'gradient_boosting',
    'k_nearest_neighbors',
    'lda',
    'liblinear_svc',
    'libsvm_svc',
    'mlp',
    'multinomial_nb',
    'passive_aggressive',
    'qda',
    'random_forest',
    'sgd'
]

feature_preprocessors = [
    'densifier',
    'extra_trees_preproc_for_classification',
    'extra_trees_preproc_for_regression',
    'fast_ica',
    'feature_agglomeration',
    'kernel_pca',
    'kitchen_sinks',
    'liblinear_svc_preprocessor',
    'no_preprocessing',
    'nystroem_sampler',
    'pca',
    'polynomial',
    'random_trees_embedding',
    'select_percentile_classification',
    'select_percentile_regression',
    'select_rates_classification',
    'select_rates_regression',
    'truncatedSVD'
]

print(len(classifiers), len(feature_preprocessors))

16 18


In [13]:
# =============================================================================
# SECTION 12 : GENERATE EXPERIMENT REGISTRY
# =============================================================================

import itertools
import pandas as pd

# ---------------------------------------------------------------------
# Classifiers
# ---------------------------------------------------------------------

CLASSIFIERS = [

    "adaboost",
    "bernoulli_nb",
    "decision_tree",
    "extra_trees",
    "gaussian_nb",
    "gradient_boosting",
    "k_nearest_neighbors",
    "lda",
    "liblinear_svc",
    "libsvm_svc",
    "multinomial_nb",
    "passive_aggressive",
    "qda",
    "random_forest",
    "sgd",
    "xgradient_boosting",

]

# ---------------------------------------------------------------------
# Feature Preprocessors
# ---------------------------------------------------------------------

FEATURE_PREPROCESSORS = [

    "densifier",
    "extra_trees_preproc_for_classification",
    "fast_ica",
    "feature_agglomeration",
    "kernel_pca",
    "kitchen_sinks",
    "liblinear_svc_preprocessor",
    "no_preprocessing",
    "nystroem_sampler",
    "pca",
    "polynomial",
    "random_trees_embedding",
    "select_percentile_classification",
    "select_rates_classification",
    "truncatedSVD",

]

# ---------------------------------------------------------------------
# Cartesian Product
# ---------------------------------------------------------------------

experiment_rows = []

experiment_id = 1

for classifier, feature_preprocessor in itertools.product(
    CLASSIFIERS,
    FEATURE_PREPROCESSORS,
):

    experiment_rows.append({

        "experiment_id": experiment_id,

        "classifier": classifier,

        "feature_preprocessor": feature_preprocessor,

        "time_budget": 3600,

        "per_run_time_limit": 300,

        "status": "Pending",

    })

    experiment_id += 1

# ---------------------------------------------------------------------
# Registry
# ---------------------------------------------------------------------

experiment_registry = pd.DataFrame(experiment_rows)

print("=" * 80)
print("Experiment Registry Created")
print("=" * 80)

print(f"Total Experiments : {len(experiment_registry)}")

display(experiment_registry.head())

Experiment Registry Created
Total Experiments : 240


,experiment_id,classifier,feature_preprocessor,time_budget,per_run_time_limit,status
0,1,adaboost,densifier,3600,300,Pending
1,2,adaboost,extra_trees_preproc_for_classification,3600,300,Pending
2,3,adaboost,fast_ica,3600,300,Pending
3,4,adaboost,feature_agglomeration,3600,300,Pending
4,5,adaboost,kernel_pca,3600,300,Pending


In [14]:
# =============================================================================
# SECTION 13 : CREATE EXPERIMENT DIRECTORIES
# =============================================================================

from pathlib import Path
import json

print("=" * 100)
print("CREATING EXPERIMENT DIRECTORIES")
print("=" * 100)

EXPERIMENT_ROOT = Path("results") / "experiments"

EXPERIMENT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# ---------------------------------------------------------------------
# Dataset IDs
# ---------------------------------------------------------------------

DATASET_IDS = [
    31,
    37,
    44,
    151,
    1461,
    4534,
]

# ---------------------------------------------------------------------
# Create folders
# ---------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    dataset_root = (
        EXPERIMENT_ROOT /
        f"dataset_{dataset_id}"
    )

    dataset_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    for _, row in experiment_registry.iterrows():

        experiment_id = int(row["experiment_id"])

        experiment_dir = (
            dataset_root /
            f"experiment_{experiment_id:03d}"
        )

        experiment_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        metadata = {

            "dataset_id": dataset_id,

            "experiment_id": experiment_id,

            "classifier": row["classifier"],

            "feature_preprocessor": row["feature_preprocessor"],

            "time_budget": row["time_budget"],

            "per_run_time_limit": row["per_run_time_limit"],

            "random_state": 42,

            "status": "Pending",

        }

        with open(
            experiment_dir / "metadata.json",
            "w",
        ) as f:

            json.dump(
                metadata,
                f,
                indent=4,
            )

        (experiment_dir / "pipelines").mkdir(
            exist_ok=True
        )

        (experiment_dir / "logs").mkdir(
            exist_ok=True
        )

        (experiment_dir / "shap").mkdir(
            exist_ok=True
        )

print()
print(f"Datasets               : {len(DATASET_IDS)}")
print(f"Experiments            : {len(experiment_registry)}")
print(f"Total Runs Created     : {len(DATASET_IDS) * len(experiment_registry)}")
print()
print("Experiment directory structure created successfully.")

CREATING EXPERIMENT DIRECTORIES

Datasets               : 6
Experiments            : 240
Total Runs Created     : 1440

Experiment directory structure created successfully.


In [ ]:
# =============================================================================
# SECTION 14A : RUN AUTOSKLEARN ON A SINGLE DATASET
# =============================================================================

import json
import pickle
import shutil
from pathlib import Path

import pandas as pd
import autosklearn.classification


def run_single_dataset(
    dataset_id,
    classifier,
    feature_preprocessor,
    experiment_id,
):

    print("=" * 100)
    print(f"Experiment : {experiment_id}")
    print(f"Dataset    : {dataset_id}")
    print(f"Classifier : {classifier}")
    print(f"Preprocessor : {feature_preprocessor}")
    print("=" * 100)

    # ------------------------------------------------------------
    # Directories
    # ------------------------------------------------------------

    experiment_dir = (
        Path("results")
        / "experiments"
        / f"dataset_{dataset_id}"
        / f"experiment_{experiment_id:03d}"
    )

    tmp_dir = experiment_dir / "tmp"

    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)

    # ------------------------------------------------------------
    # Load dataset
    # ------------------------------------------------------------

    dataset_path = DATASET_ROOT / str(dataset_id)

    features = pd.read_csv(dataset_path / "features.csv")
    targets = pd.read_csv(dataset_path / "targets.csv")

    with open(dataset_path / "categorical_indicator.json") as f:
        cat_indicators = json.load(f)

    (
        X_train,
        X_test,
        y_train,
        y_test,
        *_
    ) = ClassificationAlgorithms.prepare_data_for_SHAP_importance(
        X=features,
        y=targets,
        datapath=str(dataset_path),
        cat_indicators=cat_indicators,
        randomState=RANDOM_STATE,
        datasetId=dataset_id,
    )

    print("Data preparation completed.")

    # ------------------------------------------------------------
    # AutoSklearn
    # ------------------------------------------------------------

    automl = autosklearn.classification.AutoSklearnClassifier(

        time_left_for_this_task=3600,

        per_run_time_limit=300,

        include={
            "classifier": [classifier],
            "feature_preprocessor": [feature_preprocessor],
        },

        tmp_folder=str(tmp_dir),

        delete_tmp_folder_after_terminate=False,

        ensemble_kwargs={"ensemble_size": 1},

        initial_configurations_via_metalearning=25,

        memory_limit=10240,

        n_jobs=1,

        seed=RANDOM_STATE,
    )

    print("Starting AutoSklearn...")

    automl.fit(X_train, y_train)

    print("AutoSklearn Finished.")

    # ------------------------------------------------------------
    # Save AutoML object
    # ------------------------------------------------------------

    with open(experiment_dir / "automl.pkl", "wb") as f:
        pickle.dump(automl, f)

    # ------------------------------------------------------------
    # Save leaderboard
    # ------------------------------------------------------------

    leaderboard = automl.leaderboard()

    leaderboard.to_csv(
        experiment_dir / "leaderboard.csv",
        index=False,
    )

    # ------------------------------------------------------------
    # Update metadata
    # ------------------------------------------------------------

    metadata = {

        "experiment_id": experiment_id,

        "dataset_id": dataset_id,

        "classifier": classifier,

        "feature_preprocessor": feature_preprocessor,

        "status": "Completed",

    }

    with open(experiment_dir / "run_info.json", "w") as f:
        json.dump(metadata, f, indent=4)

    print(f"Dataset {dataset_id} completed.")

    return True

SyntaxError: unexpected EOF while parsing (3773543140.py, line 10)

In [ ]:
# =============================================================================
# SECTION 15 : RUN ALL EXPERIMENTS
# =============================================================================

from tqdm import tqdm

for experiment_id in tqdm(experiment_registry["experiment_id"]):

    try:

        run_experiment(int(experiment_id))

        experiment_registry.loc[
            experiment_registry["experiment_id"] == experiment_id,
            "status"
        ] = "Completed"

    except Exception as e:

        print(f"\nExperiment {experiment_id} FAILED")
        print(e)

        experiment_registry.loc[
            experiment_registry["experiment_id"] == experiment_id,
            "status"
        ] = "Failed"

# ------------------------------------------------------------
# Save updated registry
# ------------------------------------------------------------

experiment_registry.to_csv(
    RESULTS_DIR / "experiment_registry.csv",
    index=False,
)

print("=" * 80)
print("ALL EXPERIMENTS FINISHED")
print("=" * 80)

print(experiment_registry["status"].value_counts())

  0%|          | 0/1728 [00:00<?, ?it/s]

>>> ENTERED run_experiment
>>> Loaded experiment
Experiment ID       : 1
Dataset             : 31
Classifier          : adaboost
Preprocessor        : densifier
>>> Dataset path: datasets/31
>>> features exists : True
>>> targets exists  : True
>>> categorical exists : True
>>> Features loaded
(800, 61) (800,) (200, 61) (200,)
>>> Data prepared
X_train: (800, 61)
y_train: (800,)
>>> Starting AutoSklearn
[ERROR] [2026-08-05 04:10:36,571:Client-AutoML(42):d8792d10-9072-11f1-bd28-fb1ba8a49f58] No valid pipeline found.
Traceback (most recent call last):
  File "/home/dbanerjee/example_for_debjeet/.venv/lib/python3.8/site-packages/autosklearn/automl.py", line 751, in fit
    self.configuration_space, configspace_path = self._create_search_space(
  File "/home/dbanerjee/example_for_debjeet/.venv/lib/python3.8/site-packages/autosklearn/automl.py", line 2252, in _create_search_space
    configuration_space = pipeline.get_configuration_space(
  File "/home/dbanerjee/example_for_debjeet/.venv/li

  0%|          | 1/1728 [00:05<2:47:46,  5.83s/it]


Experiment 1 FAILED
No valid pipeline found.
>>> ENTERED run_experiment
>>> Loaded experiment
Experiment ID       : 2
Dataset             : 31
Classifier          : adaboost
Preprocessor        : extra_trees_preproc_for_classification
>>> Dataset path: datasets/31
>>> features exists : True
>>> targets exists  : True
>>> categorical exists : True
>>> Features loaded
(800, 61) (800,) (200, 61) (200,)
>>> Data prepared
X_train: (800, 61)
y_train: (800,)
>>> Starting AutoSklearn
[WARNING] [2026-08-05 04:10:44,576:Client-AutoML(42):dbf30f30-9072-11f1-bd28-fb1ba8a49f58] Capping the per_run_time_limit to 59.0 to have time for a least 2 models in each process.
[WARNING] [2026-08-05 04:10:44,939:Client-AutoMLSMBO(42)::dbf30f30-9072-11f1-bd28-fb1ba8a49f58] Configuration 601 not found
[WARNING] [2026-08-05 04:10:44,939:Client-AutoMLSMBO(42)::dbf30f30-9072-11f1-bd28-fb1ba8a49f58] Configuration 220 not found
[WARNING] [2026-08-05 04:10:44,939:Client-AutoMLSMBO(42)::dbf30f30-9072-11f1-bd28-fb1ba8a

  0%|          | 2/1728 [02:02<34:05:19, 71.10s/it]

...

  0%|          | 3/1728 [02:09<20:04:09, 41.88s/it]


Experiment 3 FAILED
The provided component 'extra_trees_preproc_for_regression' for the key 'feature_preprocessor' in the 'include' argument is not valid. The supported components for the step 'feature_preprocessor' for this task are ['densifier', 'extra_trees_preproc_for_classification', 'fast_ica', 'feature_agglomeration', 'kernel_pca', 'kitchen_sinks', 'liblinear_svc_preprocessor', 'no_preprocessing', 'nystroem_sampler', 'pca', 'polynomial', 'random_trees_embedding', 'select_percentile_classification', 'select_rates_classification', 'truncatedSVD']
>>> ENTERED run_experiment
>>> Loaded experiment
Experiment ID       : 4
Dataset             : 31
Classifier          : adaboost
Preprocessor        : fast_ica
>>> Dataset path: datasets/31
>>> features exists : True
>>> targets exists  : True
>>> categorical exists : True
>>> Features loaded
(800, 61) (800,) (200, 61) (200,)
>>> Data prepared
X_train: (800, 61)
y_train: (800,)
>>> Starting AutoSklearn
[WARNING] [2026-08-05 04:12:46,148:

Process ForkProcess-327:
Traceback (most recent call last):
  File "/usr/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/usr/lib/python3.8/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/dbanerjee/example_for_debjeet/.venv/lib/python3.8/site-packages/autosklearn/util/logging_.py", line 317, in start_log_server
    receiver.serve_until_stopped()
  File "/home/dbanerjee/example_for_debjeet/.venv/lib/python3.8/site-packages/autosklearn/util/logging_.py", line 347, in serve_until_stopped
    rd, wr, ex = select.select([self.socket.fileno()], [], [], self.timeout)
KeyboardInterrupt
  0%|          | 4/1728 [04:05<34:04:06, 71.14s/it]

>>> AutoSklearn Finished
>>> Model saved
>>> Leaderboard saved
>>> Finished successfully
>>> ENTERED run_experiment
>>> Loaded experiment
Experiment ID       : 5
Dataset             : 31
Classifier          : adaboost
Preprocessor        : feature_agglomeration
>>> Dataset path: datasets/31
>>> features exists : True
>>> targets exists  : True
>>> categorical exists : True
>>> Features loaded
(800, 61) (800,) (200, 61) (200,)
>>> Data prepared
X_train: (800, 61)
y_train: (800,)
>>> Starting AutoSklearn
[WARNING] [2026-08-05 04:14:42,197:Client-AutoML(42):6af000c7-9073-11f1-bd28-fb1ba8a49f58] Capping the per_run_time_limit to 59.0 to have time for a least 2 models in each process.
[WARNING] [2026-08-05 04:14:42,566:Client-AutoMLSMBO(42)::6af000c7-9073-11f1-bd28-fb1ba8a49f58] Configuration 601 not found
[WARNING] [2026-08-05 04:14:42,566:Client-AutoMLSMBO(42)::6af000c7-9073-11f1-bd28-fb1ba8a49f58] Configuration 220 not found
[WARNING] [2026-08-05 04:14:42,566:Client-AutoMLSMBO(42)::6af00

Process ForkProcess-399:
Traceback (most recent call last):
  File "/usr/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/usr/lib/python3.8/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/dbanerjee/example_for_debjeet/.venv/lib/python3.8/site-packages/autosklearn/util/logging_.py", line 317, in start_log_server
    receiver.serve_until_stopped()
  File "/home/dbanerjee/example_for_debjeet/.venv/lib/python3.8/site-packages/autosklearn/util/logging_.py", line 347, in serve_until_stopped
    rd, wr, ex = select.select([self.socket.fileno()], [], [], self.timeout)
KeyboardInterrupt


In [27]:
print(type(automl))

print(dir(automl))

print(dir(automl.automl_))

<class 'autosklearn.estimators.AutoSklearnClassifier'>
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_check_n_features', '_estimator_type', '_get_automl_class', '_get_param_names', '_get_tags', '_leaderboard_columns', '_more_tags', '_n_jobs', '_repr_html_', '_repr_html_inner', '_repr_mimebundle_', '_resolve_ensemble_class', '_validate_data', 'allow_string_features', 'automl_', 'build_automl', 'cv_results_', 'dask_client', 'dataset_compression', 'delete_tmp_folder_after_terminate', 'disable_evaluator_output', 'ensemble_class', 'ensemble_kwargs', 'ensemble_nbest', 'ensemble_size', 'exclude', 'fANOVA_input_', 'fit', 'fit_ensemble', 'fit_pipeline', 'get_configurat

In [28]:
print(hasattr(automl.automl_, "models_"))
print(hasattr(automl.automl_, "runhistory_"))
print(hasattr(automl.automl_, "cv_results_"))

False
False
False


In [29]:
for x in dir(automl.automl_):
    print(x)

__bool__
__class__
__delattr__
__dir__
__doc__
__eq__
__format__
__ge__
__getattribute__
__gt__
__hash__
__init__
__init_subclass__
__le__
__lt__
__ne__
__new__
__reduce__
__reduce_ex__
__repr__
__setattr__
__sizeof__
__str__
__subclasshook__


In [30]:
print(type(automl))
print(type(automl.automl_))
print(automl.automl_)

<class 'autosklearn.estimators.AutoSklearnClassifier'>
<class 'NoneType'>
None


In [31]:
print(automl.__dict__.keys())

dict_keys(['time_left_for_this_task', 'per_run_time_limit', 'initial_configurations_via_metalearning', 'ensemble_class', 'ensemble_size', 'ensemble_kwargs', 'ensemble_nbest', 'max_models_on_disc', 'seed', 'memory_limit', 'include', 'exclude', 'resampling_strategy', 'resampling_strategy_arguments', 'tmp_folder', 'delete_tmp_folder_after_terminate', 'n_jobs', 'dask_client', 'disable_evaluator_output', 'get_smac_object_callback', 'smac_scenario_args', 'logging_config', 'metadata_directory', 'metric', 'scoring_functions', 'load_models', 'get_trials_callback', 'dataset_compression', 'allow_string_features', 'automl_', '_n_jobs'])


In [32]:
import autosklearn.classification

automl = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=60,
    per_run_time_limit=30,
    include={
        "classifier": ["random_forest"],
        "feature_preprocessor": ["no_preprocessing"],
    },
    n_jobs=1,
    seed=42,
)

automl.fit(X_train, y_train)

print("FIT FINISHED")
print(type(automl.automl_))

OpenBLAS blas_thread_init: pthread_create failed for thread 15 of 64: Resource temporarily unavailable
OpenBLAS blas_thread_init: RLIMIT_NPROC 6189559 current, 6189559 max
OpenBLAS blas_thread_init: pthread_create failed for thread 16 of 64: Resource temporarily unavailable
OpenBLAS blas_thread_init: RLIMIT_NPROC 6189559 current, 6189559 max
OpenBLAS blas_thread_init: pthread_create failed for thread 17 of 64: Resource temporarily unavailable
OpenBLAS blas_thread_init: RLIMIT_NPROC 6189559 current, 6189559 max
OpenBLAS blas_thread_init: pthread_create failed for thread 18 of 64: Resource temporarily unavailable
OpenBLAS blas_thread_init: RLIMIT_NPROC 6189559 current, 6189559 max
OpenBLAS blas_thread_init: pthread_create failed for thread 19 of 64: Resource temporarily unavailable
OpenBLAS blas_thread_init: RLIMIT_NPROC 6189559 current, 6189559 max
OpenBLAS blas_thread_init: pthread_create failed for thread 20 of 64: Resource temporarily unavailable
OpenBLAS blas_thread_init: RLIMIT_NPR

[ERROR] [2026-08-05 04:46:58,806:Client-AutoML(42):ec0e0985-9077-11f1-ad17-fb1ba8a49f58] (' Dummy prediction failed with run state StatusType.CRASHED and additional output: {\'error\': \'Result queue is empty\', \'exit_status\': "<class \'pynisher.limit_function_call.AnythingException\'>", \'subprocess_stdout\': \'\', \'subprocess_stderr\': \'Process pynisher function call:\\nTraceback (most recent call last):\\n  File "/usr/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap\\n    self.run()\\n  File "/usr/lib/python3.8/multiprocessing/process.py", line 108, in run\\n    self._target(*self._args, **self._kwargs)\\n  File "/home/dbanerjee/example_for_debjeet/.venv/lib/python3.8/site-packages/pynisher/limit_function_call.py", line 133, in subprocess_func\\n    return_value = ((func(*args, **kwargs), 0))\\n  File "/home/dbanerjee/example_for_debjeet/.venv/lib/python3.8/site-packages/autosklearn/evaluation/__init__.py", line 55, in fit_predict_try_except_decorator\\n    ret

ValueError: (' Dummy prediction failed with run state StatusType.CRASHED and additional output: {\'error\': \'Result queue is empty\', \'exit_status\': "<class \'pynisher.limit_function_call.AnythingException\'>", \'subprocess_stdout\': \'\', \'subprocess_stderr\': \'Process pynisher function call:\\nTraceback (most recent call last):\\n  File "/usr/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap\\n    self.run()\\n  File "/usr/lib/python3.8/multiprocessing/process.py", line 108, in run\\n    self._target(*self._args, **self._kwargs)\\n  File "/home/dbanerjee/example_for_debjeet/.venv/lib/python3.8/site-packages/pynisher/limit_function_call.py", line 133, in subprocess_func\\n    return_value = ((func(*args, **kwargs), 0))\\n  File "/home/dbanerjee/example_for_debjeet/.venv/lib/python3.8/site-packages/autosklearn/evaluation/__init__.py", line 55, in fit_predict_try_except_decorator\\n    return ta(queue=queue, **kwargs)\\n  File "/home/dbanerjee/example_for_debjeet/.